# Optimize Refractive Index and Radius

Uses **JAX automatic differentiation** with an Adam optimizer to maximize/minimize
transmission through a slab of dielectric cylinders by tuning:

- **A)** Refractive index $n$ (with radius fixed)
- **B)** Radius $r$ (with $n$ fixed)

**Key speedup:** The T-matrix (translation matrix) depends only on cylinder positions,
NOT on $n$ or $r$. We precompute it once, then each subsequent evaluation only
recomputes Mie coefficients + linear algebra (~25x speedup).

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

sys.path.insert(0, '../..')

from Scattering_Code.smatrix_parameters import smatrix_parameters
from Scattering_Code.smatrix import smatrix_precompute, smatrix_from_precomputed

In [3]:
WAVELENGTH = 0.93
PERIOD     = 12.81
RADIUS     = 0.25
N_CYL_REF  = 1.3
MU         = 1.0
CMMAX      = 5
PHIINC     = np.pi / 2
Eva_TOL    = 1e-2
NUM_CYL    = 10
SEED_CLOCS = 42
SEED_X     = 7

n_prop = int(np.floor(PERIOD / WAVELENGTH))
n_eva  = max(int(np.floor(
    PERIOD / (2*np.pi) * np.sqrt(
        (np.log(Eva_TOL) / (2*RADIUS))**2 + (2*np.pi/WAVELENGTH)**2
    )
)) - n_prop, 0)
nmax = n_prop + n_eva
nm   = 2 * nmax + 1

## 1. Setup: Place Cylinders and Precompute T-Matrix

In [4]:
spacing = 2.5 * RADIUS
cyls_per_row = int(PERIOD / spacing)
rows_needed = NUM_CYL / cyls_per_row + 2
thickness = round(max(0.5, rows_needed * spacing * 1.5), 1)

rng = np.random.RandomState(SEED_CLOCS)
margin = RADIUS * 1.5
min_sep = 2.5 * RADIUS
clocs = np.zeros((NUM_CYL, 2))
for i in range(NUM_CYL):
    for _ in range(10000):
        x = margin + rng.rand() * (PERIOD - 2*margin)
        y = margin + rng.rand() * (thickness - 2*margin)
        if i == 0 or np.all(np.sqrt((x - clocs[:i, 0])**2 + (y - clocs[:i, 1])**2) > min_sep):
            clocs[i] = [x, y]
            break

cmmaxs = np.full(NUM_CYL, CMMAX, dtype=int)
sp = smatrix_parameters(WAVELENGTH, PERIOD, PHIINC,
                        1e-11, 1e-4, 5, 3, 1000, 3, 5, 1, PERIOD/120)

# Fixed random input vector for objective
rng_x = np.random.RandomState(SEED_X)
x_vec = rng_x.randn(nm) + 1j * rng_x.randn(nm)
x_vec = x_vec / np.linalg.norm(x_vec)

print(f"Precomputing T-matrix (this is the expensive step)...")
t0 = time.time()
precomp = smatrix_precompute(clocs, cmmaxs, PERIOD, WAVELENGTH, nmax, thickness, sp)
print(f"T-matrix precomputed in {time.time()-t0:.1f}s")

Precomputing T-matrix (this is the expensive step)...


NameError: name 'smatrix_precompute' is not defined

## 2. Define Objective: $f(\text{param}) = \|S_{21}(\text{param}) \cdot x\|^2$

In [ ]:
def objective_n(n_val):
    """Transmission through slab as function of refractive index."""
    eps = n_val**2
    cepmus_j = jnp.column_stack([jnp.full(NUM_CYL, eps), jnp.full(NUM_CYL, MU)])
    crads_j  = jnp.full(NUM_CYL, RADIUS)
    S = smatrix_from_precomputed(precomp, cmmaxs, cepmus_j, crads_j, WAVELENGTH, 'On')
    S21 = S[nm:, :nm]
    return jnp.sum(jnp.abs(S21 @ x_vec)**2).real

def objective_r(r_val):
    """Transmission through slab as function of radius."""
    cepmus_j = jnp.column_stack([jnp.full(NUM_CYL, N_CYL_REF**2), jnp.full(NUM_CYL, MU)])
    crads_j  = jnp.full(NUM_CYL, r_val)
    S = smatrix_from_precomputed(precomp, cmmaxs, cepmus_j, crads_j, WAVELENGTH, 'On')
    S21 = S[nm:, :nm]
    return jnp.sum(jnp.abs(S21 @ x_vec)**2).real

grad_n = jax.grad(objective_n)
grad_r = jax.grad(objective_r)

print(f"f(n={N_CYL_REF}) = {objective_n(jnp.float64(N_CYL_REF)):.6f}")
print(f"f(r={RADIUS})   = {objective_r(jnp.float64(RADIUS)):.6f}")

## 3. Adam Optimizer

In [ ]:
def adam_optimize(grad_fn, obj_fn, x0, lr=0.01, steps=50, beta1=0.9, beta2=0.999, eps=1e-8):
    """Simple Adam optimizer."""
    x = jnp.float64(x0)
    m, v = 0.0, 0.0
    history = {'x': [float(x)], 'f': [float(obj_fn(x))]}
    
    for t in range(1, steps+1):
        g = float(grad_fn(x))
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        x = x + lr * m_hat / (np.sqrt(v_hat) + eps)  # maximize
        x = jnp.clip(x, 1.01, 5.0)  # bounds
        history['x'].append(float(x))
        history['f'].append(float(obj_fn(x)))
        if t % 10 == 0:
            print(f"  step {t}: x={float(x):.4f}, f={history['f'][-1]:.6f}")
    
    return history

## 4. Optimize Refractive Index

In [ ]:
print("Optimizing refractive index...")
hist_n = adam_optimize(grad_n, objective_n, N_CYL_REF, lr=0.01, steps=50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_n['x'], 'b-o', ms=3)
ax1.set_xlabel('Step'); ax1.set_ylabel('n'); ax1.set_title('Refractive Index')
ax2.plot(hist_n['f'], 'r-o', ms=3)
ax2.set_xlabel('Step'); ax2.set_ylabel('f(n)'); ax2.set_title('Objective (transmission)')
plt.tight_layout()
plt.savefig('optimize_n_results.png', dpi=150)
plt.show()

## 5. Optimize Radius

In [ ]:
print("Optimizing radius...")
hist_r = adam_optimize(grad_r, objective_r, RADIUS, lr=0.005, steps=50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(hist_r['x'], 'b-o', ms=3)
ax1.set_xlabel('Step'); ax1.set_ylabel('r'); ax1.set_title('Radius')
ax2.plot(hist_r['f'], 'r-o', ms=3)
ax2.set_xlabel('Step'); ax2.set_ylabel('f(r)'); ax2.set_title('Objective (transmission)')
plt.tight_layout()
plt.savefig('optimize_r_results.png', dpi=150)
plt.show()